# Prepare NC school MMR coverage → imuGAP inputs

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

STATE = "nc"
GRADES = ["K", "1", "2", "3", "4", "5"]
AGE_OF = {g: 5 + i for i, g in enumerate(GRADES)}   # K=5 ... 5=10
MAX_AGE = max(AGE_OF.values())
DOSE = 2                                            # only 2-dose MMR is observed
MIN_ENROLLMENT = 12                                 # so no grade rounds to n = 0

ROOT_DIR = Path.cwd()
while not (ROOT_DIR / "data" / "all-schools.csv").exists() and ROOT_DIR != ROOT_DIR.parent:
    ROOT_DIR = ROOT_DIR.parent
RAW = ROOT_DIR / "data" / "all-schools.csv"
OUT = ROOT_DIR / "data" / "derived"
OUT.mkdir(exist_ok=True)
ROOT_ID = STATE.upper()

# 1. Load & filter

In [2]:
# Keep schools with coverage data (`no_data == 0`), all six grades reported, and enough enrollment that no grade rounds to `n = 0`.
cov_cols = [f"coverage_{g}" for g in GRADES]

df = pd.read_csv(RAW)
df = df[df.state == STATE].copy()
keep = (df.no_data == 0) & (df.enrollment >= MIN_ENROLLMENT) & df[cov_cols].notna().all(axis=1)
df = df[keep].sort_values("school_id").reset_index(drop=True)

df["loc_id"] = f"{STATE}_" + df.school_id.astype(str)
df["sample_n"] = np.round(df.enrollment / len(GRADES)).astype(int)
len(df)

1626

## Locations (state → county → school)

In [3]:
# imuGAP requires exactly three layers, so one fit covers one state. The root's `parent_id` is `NA`.
counties = sorted(df.county.unique())
locations = pd.concat([
    pd.DataFrame({"loc_id": [ROOT_ID], "parent_id": [np.nan]}),
    pd.DataFrame({"loc_id": counties, "parent_id": ROOT_ID}),
    pd.DataFrame({"loc_id": df.loc_id, "parent_id": df.county}),
], ignore_index=True)
locations.shape

(1727, 2)

## Observations & populations

One observation per school × grade. `age` is the real age (K=5 … 5=10),
`cohort = MAX_AGE − age + 1`, and `dose = 2`. `positive = round(coverage × n)`,
clamped to `n`. Rows are grade-major (all kindergarten first, then grade 1, …)
to match the canonical ordering.

In [4]:
blocks = []
for g in GRADES:
    age = AGE_OF[g]
    sub = df[["loc_id", "sample_n"]].copy()
    sub["age"] = age
    sub["cohort"] = MAX_AGE - age + 1
    sub["positive"] = np.minimum(sub.sample_n, np.round(df[f"coverage_{g}"] * sub.sample_n)).astype(int)
    blocks.append(sub)

long = pd.concat(blocks, ignore_index=True)
long["obs_id"] = np.arange(1, len(long) + 1)
long["dose"] = DOSE
long["weight"] = 1

observations = long[["obs_id", "positive", "sample_n"]]
populations = long[["obs_id", "loc_id", "cohort", "age", "dose", "weight"]]
observations.shape, populations.shape

((9756, 3), (9756, 6))

## Validate (structural)

Mirror imuGAP's canonicalizer checks on the Python side before writing.

In [5]:
assert observations.obs_id.is_unique
assert (observations.sample_n >= 1).all()
assert (observations.positive >= 0).all()
assert (observations.sample_n >= observations.positive).all()
assert populations.dose.between(1, 2).all()
assert (populations.cohort >= 1).all() and (populations.age >= 1).all()
assert set(populations.loc_id) <= set(locations.loc_id)
assert populations.obs_id.isin(observations.obs_id).all()
assert locations.parent_id.isna().sum() == 1
assert locations.loc_id.is_unique

n_schools = locations.loc_id.str.startswith(f"{STATE}_").sum()
print(f"structural checks passed: {n_schools} schools, {len(observations)} observations")

structural checks passed: 1626 schools, 9756 observations


## Write

In [6]:
tables = {
    f"{STATE}_locations": locations,
    f"{STATE}_observations": observations,
    f"{STATE}_populations": populations,
}
for name, tbl in tables.items():
    path = OUT / f"{name}.csv"
    tbl.to_csv(path, index=False)
    print("wrote", path)

wrote /Users/0ssamaak0/Documents/mmed/VPD Immunity Estimation/data/derived/nc_locations.csv
wrote /Users/0ssamaak0/Documents/mmed/VPD Immunity Estimation/data/derived/nc_observations.csv
wrote /Users/0ssamaak0/Documents/mmed/VPD Immunity Estimation/data/derived/nc_populations.csv
